In [1]:
import oracledb
import pandas as pd
import torch
import random

oracledb.init_oracle_client(lib_dir=r"D:\\instantclient_23_9")

conn = oracledb.connect(
    user="adsql",          # 사용자명
    password="oracle_4U",      # 비밀번호
    dsn="localhost:1521/xe" # 접속 정보 (SQL Developer와 동일)
)
cur = conn.cursor()

In [10]:
from prediction_all import * 

cur.execute("TRUNCATE TABLE USER_INPUT")
conn.commit()

user_input = 'caffeine' # 입력 (실제는 사용자한테 분자 이름 받아야 함)
user_input_info = return_chembl_data(user_input) # 리스트로 나옴

# 입력 받은 분자 이름으로 chembl에서 정보 찾기
sql = "select * from user_input where rownum <= 1" 
select_user_input = pd.read_sql(sql, conn)
user_input_col = list(select_user_input.columns)
user_input_data = return_chembl_data(user_input)
insert_data('user_input', user_input_col, user_input_data) # 사용자가 입력한 분자 DB에 저장

return_value = dict(zip(user_input_col, user_input_data))

# 사용자가 입력한 분자로 예측 수행하기
select_user_gen = "select * from user_generative where rownum <= 1" 
user_gen_db = pd.read_sql(select_user_gen, conn)
user_gen_col = list(user_gen_db.columns)
user_generate_molecule_result = [] # 새로 만들어진 분자 

for j in range(5):
	cur.execute("SELECT SEQ_USER_GEN.NEXTVAL FROM DUAL")
	seq_user_gen_val = cur.fetchone()[0]

	torch.manual_seed(torch.randint(0, 1000000, (1,)).item())
	user_gen_data = [return_value['U_CHEMBL_ID']] + [f"UNEW_MOLECULE{seq_user_gen_val}"] + make_smiles(return_value['U_CANOSMILES'])
	user_generate_molecule_result.append(user_gen_data)

for user_gen_row in user_generate_molecule_result:
	pki = list(predict_pKi(user_gen_row[2]))
	pkd = list(predict_pKd(user_gen_row[2]))
	toxic = [toxic_predict(user_gen_row[2])]
	user_gen_data2 = user_gen_row + pki + pkd + toxic

	user_gen_dic = dict(zip(user_gen_col, user_gen_data2))

	columns = ', '.join(user_gen_col)
	placeholders = ', '.join([f':{k}' for k in user_gen_col])

	insert_user_gen_sql = f"INSERT INTO USER_GENERATIVE ({columns}) VALUES ({placeholders})"

	cur.execute(insert_user_gen_sql, user_gen_dic)
	print('집어넣기 성공')
	conn.commit()

C:\Users\amysm\AppData\Local\Temp\ipykernel_10376\3868477477.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  select_user_input = pd.read_sql(sql, conn)
C:\Users\amysm\AppData\Local\Temp\ipykernel_10376\3868477477.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  user_gen_db = pd.read_sql(select_user_gen, conn)
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
c:\Users\amysm\AppData\Local\Programs\Python\Python31

집어넣기 성공


c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


집어넣기 성공


c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


집어넣기 성공


c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


집어넣기 성공
집어넣기 성공


c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(


In [ ]:
user_selected_button = '암 치료제' # 누른 버튼 이름
sql = f"SELECT * FROM disease_input where d_category = '{user_selected_button}'"
# 암 치료제 등 버튼 누르면 disease_input에서 해당 카테고리의 smiles만 가져옴
res = pd.read_sql(sql, conn)
randoms = [0,1]
orig_molecule = res.iloc[randoms].to_dict(orient='records') # 카테고리별 smiles 2개 관련 정보

# orig_molecule로 새로운 분자 생성 후에 DB에 저장하는 거까지
molecule_id_number = 0
db_col = ['DNEW_CHEMBL_ID', 'DNEW_NAME', 'DNEW_CANOSMILES', 'DNEW_IMAGE_BASE64',
		'DNEW_MOL_WEIGHT', 'DNEW_LOGP', 'DNEW_QED', 'DNEW_HBD', 'DNEW_HBA',
		'DNEW_PKI_RES', 'DNEW_PKI', 'DNEW_PKD_RES', 'DNEW_PKD', 'DNEW_TOXIC',
		'DNEW_CATEGORY']

generate_molecule_result = []

for i in orig_molecule:
	for j in range(5):
		cur.execute("SELECT SEQ_DISEASE_GEN.NEXTVAL FROM DUAL")
		seq_val = cur.fetchone()[0]

		torch.manual_seed(torch.randint(0, 1000000, (1,)).item())
		res = [i['D_CHEMBL_ID']] + [f"DNEW_MOLECULE{seq_val}"] + make_smiles(i['D_CANOSMILES'])
		generate_molecule_result.append(res)
		molecule_id_number += 1
  
for res in generate_molecule_result:
	pki = list(predict_pKi(res[2]))
	pkd = list(predict_pKd(res[2]))
	toxic = [toxic_predict(res[2])]
	res2 = res + pki + pkd + toxic + [i['D_CATEGORY']]
 
	dic = dict(zip(db_col, res2))

	columns = ', '.join(dic.keys())
	placeholders = ', '.join([f':{k}' for k in dic.keys()])

	sql = f"INSERT INTO DISEASE_GENERATIVE ({columns}) VALUES ({placeholders})"

	cur.execute(sql, dic)
	conn.commit()

C:\Users\amysm\AppData\Local\Temp\ipykernel_12088\1206953691.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  res = pd.read_sql(sql, conn)
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
c:\Users\amysm\AppData\Local\Programs\Python\Python310\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but PCA was fitted with feature names
  warnings.warn(
c:\Users\amysm\AppData\Local\Progra